In [21]:
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import os
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,confusion_matrix,accuracy_score

In [22]:
data_files = ['UNSW-NB15_1.csv', 'UNSW-NB15_2.csv', 'UNSW-NB15_3.csv', 'UNSW-NB15_4.csv']
dataframes = [pd.read_csv(file, header=None, low_memory=False) for file in data_files]
df = pd.concat(dataframes, ignore_index=True)
print("Merged Data Shape:", df.shape)


Merged Data Shape: (2540047, 49)


In [23]:
features=pd.read_csv('NUSW-NB15_features.csv',encoding='latin1')
features.head(49)


,No.,Name,Type,Description
0,1,srcip,nominal,Source IP address
1,2,sport,integer,Source port number
2,3,dstip,nominal,Destination IP address
3,4,dsport,integer,Destination port number
4,5,proto,nominal,Transaction protocol
5,6,state,nominal,Indicates to the state and its dependent proto...
6,7,dur,Float,Record total duration
7,8,sbytes,Integer,Source to destination transaction bytes
8,9,dbytes,Integer,Destination to source transaction bytes
9,10,sttl,Integer,Source to destination time to live value


In [24]:
import pandas as pd

# ✅ Step 1: Correct column names list (49 expected columns)
column_names = [
    'srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes',
    'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts',
    'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len',
    'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat',
    'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd',
    'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm',
    'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'label'
]
# ✅ Step 2: Read and merge CSVs (assuming data_files is already defined)
dataframes = [pd.read_csv(file, header=None, low_memory=False) for file in data_files]
df = pd.concat(dataframes, ignore_index=True)

# ✅ Step 3: Assign column names
df.columns = column_names

# ✅ Step 4: Confirm
print("Final dataset shape:", df.shape)
print("Final column count:", len(df.columns))
print(" Column names:")
print(df.columns.tolist())
import pandas as pd

# Fix column names
df.columns = df.columns.str.strip().str.lower()


Final dataset shape: (2540047, 49)
Final column count: 49
 Column names:
['srcip', 'sport', 'dstip', 'dsport', 'proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'Sload', 'Dload', 'Spkts', 'Dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'Sjit', 'Djit', 'Stime', 'Ltime', 'Sintpkt', 'Dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat', 'label']


In [25]:
# 🧹 Step 3: Drop specified columns only
columns_to_drop = ['id', 'srcip', 'sport', 'dstip', 'dsport', 'label']
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')


In [26]:
# Save the original columns
original_columns = df.columns.tolist()

# Drop specified columns
columns_to_drop = ['id', 'srcip', 'sport', 'dstip', 'dsport', 'label']
df.drop(columns=columns_to_drop, inplace=True, errors='ignore')

# Compare columns
removed_columns = [col for col in columns_to_drop if col in original_columns]

print("✅ Remaining columns:", df.columns.tolist())


✅ Remaining columns: ['proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload', 'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime', 'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_flw_http_mthd', 'is_ftp_login', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'attack_cat']


In [27]:
df.head()

,proto,state,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,service,...,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,attack_cat
0,udp,CON,0.001055,132,164,31,29,0,0,dns,...,0.0,0,3,7,1,3,1,1,1,NaN
1,udp,CON,0.036133,528,304,31,29,0,0,-,...,0.0,0,2,4,2,3,1,1,2,NaN
2,udp,CON,0.001119,146,178,31,29,0,0,dns,...,0.0,0,12,8,1,2,2,1,1,NaN
3,udp,CON,0.001209,132,164,31,29,0,0,dns,...,0.0,0,6,9,1,1,1,1,1,NaN
4,udp,CON,0.001169,146,178,31,29,0,0,dns,...,0.0,0,7,9,1,1,1,1,1,NaN


In [28]:
print("Number of rows:",df.shape[0])

Number of rows: 2540047


In [29]:
print("Number of columns:",df.shape[1])

Number of columns: 44


In [30]:
print("Shape of the dataset:",df.shape)

Shape of the dataset: (2540047, 44)


In [31]:

null_counts = df.isnull().sum().sort_values(ascending=False)
print("Top 3 columns with most null values:")
print(null_counts.head(3))

top_null_columns = null_counts.head(3).index.tolist()
df.drop(columns=top_null_columns,inplace=True)
print(f"\nDropped columns:{top_null_columns}")
print("Shape after dropping 3 columns with most nulls:",df.shape)

df.dropna(inplace=True)
print("Dataset shape after dropping any remaining nulls rows:",df.shape)

print(df.isnull().sum())
print("Total null values:",df.isnull().sum().sum())


Top 3 columns with most null values:
attack_cat          2218764
is_ftp_login        1429879
ct_flw_http_mthd    1348145
dtype: int64

Dropped columns:['attack_cat', 'is_ftp_login', 'ct_flw_http_mthd']
Shape after dropping 3 columns with most nulls: (2540047, 41)
Dataset shape after dropping any remaining nulls rows: (2540047, 41)
proto               0
state               0
dur                 0
sbytes              0
dbytes              0
sttl                0
dttl                0
sloss               0
dloss               0
service             0
sload               0
dload               0
spkts               0
dpkts               0
swin                0
dwin                0
stcpb               0
dtcpb               0
smeansz             0
dmeansz             0
trans_depth         0
res_bdy_len         0
sjit                0
djit                0
stime               0
ltime               0
sintpkt             0
dintpkt             0
tcprtt              0
synack              0
ackdat 

In [33]:
df.drop_duplicates(inplace=True)
print("Dataset shape after removing duplicates:",df.shape)

Dataset shape after removing duplicates: (2046657, 41)


In [32]:
# 🔄 Step 4: Encode categorical features
cat_cols = df.select_dtypes(include='object').columns
le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))


In [34]:
print("Columns in df:", df.columns.tolist())

Columns in df: ['proto', 'state', 'dur', 'sbytes', 'dbytes', 'sttl', 'dttl', 'sloss', 'dloss', 'service', 'sload', 'dload', 'spkts', 'dpkts', 'swin', 'dwin', 'stcpb', 'dtcpb', 'smeansz', 'dmeansz', 'trans_depth', 'res_bdy_len', 'sjit', 'djit', 'stime', 'ltime', 'sintpkt', 'dintpkt', 'tcprtt', 'synack', 'ackdat', 'is_sm_ips_ports', 'ct_state_ttl', 'ct_ftp_cmd', 'ct_srv_src', 'ct_srv_dst', 'ct_dst_ltm', 'ct_src_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm']
